In [ ]:
from dinov2.data.datasets.s2_csv import S2CsvDataset
from torchvision import transforms
from torch.utils.data import DataLoader

# 先算一次均值方差（可选）
# from dinov2.utils.data import compute_dataset_stats
# tmp_ds = S2CsvDataset(csv_path="data_csv/96_3_train.csv", scale_to_unit=True, pad_to_multiple=14, normalize_stats=None)
# compute_dataset_stats(tmp_ds, subset=0.1)  # 输出 mean/std，填到 normalize_stats

train_tf = transforms.Compose([
    # 这里可以插入你需要的随机裁剪/翻转
])

ds_train = S2CsvDataset(
    csv_path="data_csv/96_3_train.csv",
    ds_cfg_name="s2_12band",
    normalize_stats=None,   # 如果已有 mean/std，填为 (mean, std)
    scale_to_unit=True,     # uint16 -> [0,1]
    pad_to_multiple=14,     # 96 -> 98，避免 patch 截断
    transform=train_tf,
)

ds_test = S2CsvDataset(
    csv_path="data_csv/96_3_test.csv",
    ds_cfg_name="s2_12band",
    normalize_stats=None,
    scale_to_unit=True,
    pad_to_multiple=14,
    transform=None,
)

train_loader = DataLoader(ds_train, batch_size=64, shuffle=True, num_workers=8)
test_loader = DataLoader(ds_test, batch_size=64, shuffle=False, num_workers=8)


In [2]:
from dinov2.data.datasets.s2_csv import S2CsvDataset

csv = "data_csv/96_3_train.csv"
subset = 500  # 2% 抽样，可改为 int 比如 500
ds = S2CsvDataset(
    csv_path=csv,
    ds_cfg_name="s2_12band",
    scale_to_unit=True,
    compute_stats=True,
    compute_stats_subset=subset,
    pad_to_multiple=None,  # 统计时不需要 padding
)

mean, std = ds.get_normalize_stats()
print("mean =", mean.tolist())
print("std  =", std.tolist())


mean = [0.029740596190094948, 0.03309573978185654, 0.03938313201069832, 0.04678337648510933, 0.05201242119073868, 0.05771360546350479, 0.06091780215501785, 0.06377039849758148, 0.0, 0.0, 0.07156942784786224, 0.06320085376501083]
std  = [0.011487055569887161, 0.012671221047639847, 0.014261508360505104, 0.018188122659921646, 0.018463384360074997, 0.016438089311122894, 0.016748901456594467, 0.01686006784439087, 9.999999974752427e-07, 9.999999974752427e-07, 0.02173806168138981, 0.022262444719672203]


In [ ]:
import numpy as np
import tifffile

def plume_ratio(mask_path):
    m = tifffile.imread(mask_path)
    return (m > 0).mean()

# 分别算 train/test 的 label=1
mask_path="/home/yuyao/panopticon/data_csv/96_3_train.csv"
print(plume_ratio(mask_path))

In [6]:
import pandas as pd
from pathlib import Path

csv_path = Path("data_csv/hongxuan_temporal_32/test.csv")

df = pd.read_csv(csv_path)
df.to_csv(csv_path.with_suffix(".bak"), index=False)

df["plume_mask_path"] = df["plume_mask_path"].str.replace(
    r"^/home", "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao", regex=True
)

df.to_csv(csv_path, index=False)